# Polymarket Connection Checks

This notebook is reserved for Milestone 1 connectivity and payload validation.

Goals:
- verify public REST connectivity
- inspect payload shape and stable identifiers
- test WebSocket subscription flow
- record sample payloads for later schema design


## Planned checks

1. Verify Gamma `GET /markets` reachability and capture a representative market sample
2. Verify public CLOB `GET /book`, `GET /price`, and `GET /prices-history` access for one Gamma token
3. Verify public Data API leaderboard, wallet, holder, position, trade-history, and open-interest endpoints
4. Confirm the key identifiers exposed across Gamma, CLOB, and Data API payloads
5. Verify the public CLOB market-channel WebSocket subscription flow for the selected token
6. Record local sample payload locations for later schema design, including WebSocket captures


In [ ]:
from __future__ import annotations

import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import httpx
from dotenv import load_dotenv


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "TASKS.md").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root from the current working directory.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
load_dotenv(REPO_ROOT / ".env")

DEFAULT_BASE_URLS = {
    "gamma": "https://gamma-api.polymarket.com",
    "clob": "https://clob.polymarket.com",
    "data_api": "https://data-api.polymarket.com",
}
BASE_URLS = {
    "gamma": os.getenv("POLYMARKET_GAMMA_BASE_URL", DEFAULT_BASE_URLS["gamma"]),
    "clob": os.getenv("POLYMARKET_CLOB_BASE_URL", DEFAULT_BASE_URLS["clob"]),
    "data_api": os.getenv("POLYMARKET_DATA_API_BASE_URL", DEFAULT_BASE_URLS["data_api"]),
}
SAMPLE_DIRS = {
    "gamma": REPO_ROOT / "data/raw/gamma/connection_checks",
    "clob": REPO_ROOT / "data/raw/clob/connection_checks",
    "data_api": REPO_ROOT / "data/raw/data_api/connection_checks",
}

for sample_dir in SAMPLE_DIRS.values():
    sample_dir.mkdir(parents=True, exist_ok=True)


def fetch_json(url: str, params: dict[str, Any] | None = None) -> Any:
    response = httpx.get(
        url,
        params=params,
        headers={"Accept": "application/json"},
        timeout=20.0,
    )
    response.raise_for_status()
    return response.json()


def parse_clob_token_ids(raw_value: Any) -> list[str]:
    if raw_value is None:
        return []
    if isinstance(raw_value, list):
        return [str(item) for item in raw_value if str(item).strip()]
    if isinstance(raw_value, str):
        stripped = raw_value.strip()
        if not stripped:
            return []
        try:
            decoded = json.loads(stripped)
        except json.JSONDecodeError:
            return [part.strip() for part in stripped.split(",") if part.strip()]
        if isinstance(decoded, list):
            return [str(item) for item in decoded if str(item).strip()]
    raise ValueError(f"Unsupported clobTokenIds value: {raw_value!r}")


def slugify_endpoint(endpoint: str) -> str:
    return endpoint.strip("/").replace("/", "_").replace("-", "_") or "root"


def save_sample_payload(source: str, endpoint: str, params: dict[str, Any] | None, payload: Any) -> Path:
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    wrapped_payload = {
        "source": source,
        "endpoint": endpoint,
        "params": params or {},
        "fetched_at_utc": datetime.now(timezone.utc).isoformat(),
        "payload": payload,
    }
    sample_path = SAMPLE_DIRS[source] / f"{timestamp}_{slugify_endpoint(endpoint)}.json"
    sample_path.write_text(json.dumps(wrapped_payload, indent=2, sort_keys=True))
    return sample_path


def format_market_identifiers(market: dict[str, Any]) -> dict[str, Any]:
    return {
        "market_id": market.get("id"),
        "question": market.get("question"),
        "slug": market.get("slug"),
        "conditionId": market.get("conditionId") or market.get("condition_id"),
        "clobTokenIds": parse_clob_token_ids(market.get("clobTokenIds")),
    }


def extract_records(payload: Any) -> list[dict[str, Any]]:
    if isinstance(payload, list):
        records: list[dict[str, Any]] = []
        for item in payload:
            if not isinstance(item, dict):
                continue
            if isinstance(item.get("positions"), list):
                records.extend(position for position in item["positions"] if isinstance(position, dict))
                continue
            if isinstance(item.get("holders"), list):
                records.extend(holder for holder in item["holders"] if isinstance(holder, dict))
                continue
            records.append(item)
        return records
    if isinstance(payload, dict):
        return [payload]
    return []


FIELD_GROUPS = {
    "wallet_identity": ("proxyWallet", "user"),
    "side": ("side",),
    "size": ("size", "amount", "totalBought", "value"),
    "outcome": ("outcome", "outcomeIndex", "oppositeOutcome"),
    "timestamp": ("timestamp", "endDate"),
}


def summarize_observed_fields(record: dict[str, Any]) -> dict[str, list[str]]:
    return {
        label: [field_name for field_name in aliases if field_name in record]
        for label, aliases in FIELD_GROUPS.items()
    }


print("Base URLs:")
for source_name, base_url in BASE_URLS.items():
    print(f"- {source_name}: {base_url}")

print("\nSample directories:")
for source_name, sample_dir in SAMPLE_DIRS.items():
    print(f"- {source_name}: {sample_dir}")


In [ ]:
gamma_endpoint = "/markets"
gamma_params = {"limit": 10, "closed": False}
gamma_response = fetch_json(f"{BASE_URLS['gamma']}{gamma_endpoint}", params=gamma_params)
gamma_sample_path = save_sample_payload("gamma", gamma_endpoint, gamma_params, gamma_response)

if isinstance(gamma_response, dict) and "data" in gamma_response:
    gamma_markets = gamma_response["data"]
else:
    gamma_markets = gamma_response

if not isinstance(gamma_markets, list):
    raise TypeError(f"Expected the Gamma /markets response to be a list, got {type(gamma_markets)!r}")

usable_gamma_market = None
for market in gamma_markets:
    identifiers = format_market_identifiers(market)
    if identifiers["conditionId"] and identifiers["clobTokenIds"]:
        usable_gamma_market = market
        gamma_market_identifiers = identifiers
        break

if usable_gamma_market is None:
    raise RuntimeError("No Gamma market in the sample response exposed both conditionId and clobTokenIds.")

print("Selected Gamma market identifiers:")
print(json.dumps(gamma_market_identifiers, indent=2))
print(f"Saved Gamma sample: {gamma_sample_path}")


In [ ]:
if "gamma_market_identifiers" not in globals():
    raise RuntimeError("Run the Gamma market cell before the CLOB verification cell.")

condition_id = str(gamma_market_identifiers["conditionId"])
token_id = str(gamma_market_identifiers["clobTokenIds"][0])

book_endpoint = "/book"
book_params = {"token_id": token_id}
book_payload = fetch_json(f"{BASE_URLS['clob']}{book_endpoint}", params=book_params)
book_sample_path = save_sample_payload("clob", book_endpoint, book_params, book_payload)

buy_price_endpoint = "/price"
buy_price_params = {"token_id": token_id, "side": "BUY"}
buy_price_payload = fetch_json(f"{BASE_URLS['clob']}{buy_price_endpoint}", params=buy_price_params)
buy_price_sample_path = save_sample_payload("clob", buy_price_endpoint, buy_price_params, buy_price_payload)

sell_price_endpoint = "/price"
sell_price_params = {"token_id": token_id, "side": "SELL"}
sell_price_payload = fetch_json(f"{BASE_URLS['clob']}{sell_price_endpoint}", params=sell_price_params)
sell_price_sample_path = save_sample_payload("clob", sell_price_endpoint, sell_price_params, sell_price_payload)

history_endpoint = "/prices-history"
history_params = {"market": token_id, "interval": "1w", "fidelity": 5}
history_payload = fetch_json(f"{BASE_URLS['clob']}{history_endpoint}", params=history_params)
history_sample_path = save_sample_payload("clob", history_endpoint, history_params, history_payload)

book_market = str(book_payload.get("market"))
book_asset_id = str(book_payload.get("asset_id"))
condition_matches = condition_id == book_market
token_matches = token_id == book_asset_id

print("Selected Gamma and CLOB join keys:")
print(
    json.dumps(
        {
            "Gamma.conditionId": condition_id,
            "Gamma.clobTokenIds[0]": token_id,
            "CLOB book.market": book_market,
            "CLOB book.asset_id": book_asset_id,
        },
        indent=2,
    )
)
print(f"Gamma.conditionId == CLOB book.market: {condition_matches}")
print(f"Gamma.clobTokenIds[0] == CLOB book.asset_id: {token_matches}")

if not condition_matches or not token_matches:
    raise AssertionError("Gamma and CLOB identifiers did not line up as expected.")

history_points = history_payload.get("history") if isinstance(history_payload, dict) else history_payload
history_count = len(history_points) if isinstance(history_points, list) else "unknown"

print("\nSaved CLOB samples:")
for sample_path in [book_sample_path, buy_price_sample_path, sell_price_sample_path, history_sample_path]:
    print(f"- {sample_path}")

print("\nRepresentative payload notes:")
print(f"- buy price payload keys: {sorted(buy_price_payload.keys()) if isinstance(buy_price_payload, dict) else 'n/a'}")
print(f"- sell price payload keys: {sorted(sell_price_payload.keys()) if isinstance(sell_price_payload, dict) else 'n/a'}")
print(f"- prices-history point count: {history_count}")
print("- note: the live CLOB API currently requires fidelity >= 5 for a 1w history query")


In [ ]:
if "gamma_market_identifiers" not in globals():
    raise RuntimeError("Run the Gamma and CLOB verification cells before the Data API verification cell.")

condition_id = str(gamma_market_identifiers["conditionId"])
token_id = str(gamma_market_identifiers["clobTokenIds"][0])

default_sample_wallet = "0x56687bf447db6ffa42ffe2204a05edaa20f55839"
leaderboard_endpoint = "/v1/leaderboard"
leaderboard_params = {"category": "OVERALL", "timePeriod": "ALL", "orderBy": "PNL", "limit": 5}
leaderboard_payload = fetch_json(f"{BASE_URLS['data_api']}{leaderboard_endpoint}", params=leaderboard_params)
leaderboard_sample_path = save_sample_payload("data_api", leaderboard_endpoint, leaderboard_params, leaderboard_payload)
leaderboard_records = extract_records(leaderboard_payload)
leaderboard_wallet = next(
    (str(record.get("proxyWallet")) for record in leaderboard_records if record.get("proxyWallet")),
    None,
)
sample_wallet = leaderboard_wallet or default_sample_wallet
wallet_seed_source = "/v1/leaderboard" if leaderboard_wallet else "docs fallback wallet"

data_api_checks = [
    {
        "label": "wallet positions",
        "endpoint": "/positions",
        "params": {"user": sample_wallet, "limit": 5, "sortBy": "TOKENS"},
        "note": "Current position snapshot; useful for wallet holdings and market linkage, but not individual fills.",
    },
    {
        "label": "closed positions",
        "endpoint": "/closed-positions",
        "params": {"user": sample_wallet, "limit": 5, "sortBy": "TIMESTAMP"},
        "note": "Closed-position history includes outcome and timestamps, but it is position-level rather than trade-level history.",
    },
    {
        "label": "wallet activity",
        "endpoint": "/activity",
        "params": {"user": sample_wallet, "limit": 10, "type": "TRADE", "sortBy": "TIMESTAMP", "sortDirection": "DESC"},
        "note": "Best wallet-centric trade log in the public Data API; includes side, size, price, asset, and transaction hash.",
    },
    {
        "label": "market trades",
        "endpoint": "/trades",
        "params": {"market": condition_id, "limit": 10},
        "note": "Market-centric trade history still exposes proxyWallet, side, size, outcome, asset, and timestamps for attribution checks.",
    },
    {
        "label": "top holders",
        "endpoint": "/holders",
        "params": {"market": condition_id, "limit": 10},
        "note": "Useful concentration view per token, but no trade timestamps or side fields.",
    },
    {
        "label": "open interest",
        "endpoint": "/oi",
        "params": {"market": condition_id},
        "note": "Market-level size only; no wallet identities or side attribution.",
    },
]

endpoint_summaries = []
saved_paths = [leaderboard_sample_path]

for check in data_api_checks:
    payload = fetch_json(f"{BASE_URLS['data_api']}{check['endpoint']}", params=check["params"])
    sample_path = save_sample_payload("data_api", check["endpoint"], check["params"], payload)
    saved_paths.append(sample_path)
    records = extract_records(payload)
    sample_record = records[0] if records else {}
    endpoint_summaries.append(
        {
            "label": check["label"],
            "endpoint": check["endpoint"],
            "params": check["params"],
            "record_count": len(records),
            "sample_keys": sorted(sample_record.keys()),
            "observed_fields": summarize_observed_fields(sample_record),
            "sample_path": str(sample_path),
            "note": check["note"],
        }
    )

print("Selected Data API wallet seed:")
print(
    json.dumps(
        {
            "sample_wallet": sample_wallet,
            "wallet_seed_source": wallet_seed_source,
            "selected_market_conditionId": condition_id,
            "selected_market_token_id": token_id,
        },
        indent=2,
    )
)

print("\nData API endpoint coverage:")
for summary in endpoint_summaries:
    print(json.dumps(summary, indent=2))

print("\nSaved Data API samples:")
for sample_path in saved_paths:
    print(f"- {sample_path}")

print("\nUsability notes:")
for summary in endpoint_summaries:
    print(f"- {summary['label']}: {summary['note']}")


## WebSocket verification

Use the selected CLOB token from the earlier REST checks to verify that the public market channel opens, accepts a subscription, and emits messages that are understandable enough for later collection code.


In [ ]:
from src.clients.polymarket_websocket import capture_market_channel_samples

if "gamma_market_identifiers" not in globals():
    raise RuntimeError("Run the Gamma and CLOB verification cells before the WebSocket verification cell.")

ws_base_url = os.getenv("POLYMARKET_WS_URL", "wss://ws-subscriptions-clob.polymarket.com/ws")
ws_asset_ids = [str(gamma_market_identifiers["clobTokenIds"][0])]
websocket_sample_dir = REPO_ROOT / "data/raw/websocket/connection_checks"
websocket_sample_dir.mkdir(parents=True, exist_ok=True)

websocket_capture = await capture_market_channel_samples(
    ws_base_url=ws_base_url,
    asset_ids=ws_asset_ids,
    output_dir=websocket_sample_dir,
    max_messages=3,
    reconnect_attempts=1,
    message_timeout_seconds=20.0,
)

websocket_preview = [message["payload"] for message in websocket_capture["messages"][:2]]

print("WebSocket capture summary:")
print(
    json.dumps(
        {
            "ws_base_url": ws_base_url,
            "channel_url": websocket_capture["url"],
            "asset_ids": ws_asset_ids,
            "message_count": websocket_capture["message_count"],
            "message_shapes": websocket_capture["message_shapes"],
            "sample_path": websocket_capture["sample_path"],
        },
        indent=2,
    )
)

print("\nRepresentative WebSocket messages:")
for payload in websocket_preview:
    print(json.dumps(payload, indent=2))

print("\nConnection events:")
for event in websocket_capture["connection_events"]:
    print(json.dumps(event, indent=2))

print("\nReconnect and heartbeat expectations:")
print("- Reconnect: if the socket closes before enough samples arrive, rerun the same subscription and capture the close event plus the next connect attempt.")
print("- Heartbeat: the public market-channel docs do not document an application-level heartbeat, so treat idle timeouts and close events as signals to reconnect rather than assuming a documented PING/PONG message.")


## Findings template

Update this section after running the notebook locally.

- Reachable endpoints:
  - Gamma `GET /markets`
  - CLOB `GET /book`
  - CLOB `GET /price` for both `BUY` and `SELL`
  - CLOB `GET /prices-history` using `market=<token_id>`, `interval=1w`, and `fidelity=5`
  - CLOB WebSocket market channel `wss://ws-subscriptions-clob.polymarket.com/ws/market`
  - Data API `GET /v1/leaderboard`
  - Data API `GET /positions`
  - Data API `GET /closed-positions`
  - Data API `GET /activity`
  - Data API `GET /trades`
  - Data API `GET /holders`
  - Data API `GET /oi`
- Required auth or headers:
  - Gamma, CLOB public market data, and Data API checks should run with public access only.
  - The public CLOB market channel should accept an unauthenticated subscription payload containing `assets_ids`, `type`, and `custom_feature_enabled`.
- Stable identifiers to capture:
  - `Gamma.conditionId`
  - `Gamma.clobTokenIds[0]`
  - `CLOB book.market`
  - `CLOB book.asset_id`
  - `WebSocket asset_id`
  - `WebSocket event_type`
  - `Data API proxyWallet`
  - `Data API conditionId` and `market`
  - `Data API asset` and `token`
- Field coverage to confirm:
  - wallet identity: `proxyWallet`
  - side: `activity.side`, `trades.side`
  - size: `positions.size`, `activity.size`, `trades.size`, `holders.amount`, `oi.value`
  - outcome: `positions.outcome`, `closed-positions.outcome`, `activity.outcome`, `trades.outcome`
  - timestamps: `activity.timestamp`, `trades.timestamp`, `closed-positions.timestamp`, `closed-positions.endDate`
  - websocket market shape: top-level keys grouped by `event_type`
- Saved sample locations:
  - `data/raw/gamma/connection_checks/`
  - `data/raw/clob/connection_checks/`
  - `data/raw/websocket/connection_checks/`
  - `data/raw/data_api/connection_checks/`
- Known gaps or blockers:
  - `/holders` is a concentration snapshot, not a trade log.
  - `/oi` is market-level only and does not expose wallet attribution.
  - `/positions` and `/closed-positions` summarize positions, so trade sequencing still depends on `/activity` or `/trades`.
  - The public market-channel docs do not currently document an application-level heartbeat, so reconnect behavior should be verified from observed close and timeout events.
